- Normalizing audio data
- Cutting into specific length audio chunks

In [ ]:
import os
import pandas as pd
import numpy as np
from pydub import AudioSegment
import math
import soundfile as sf
import librosa
from pathlib import Path
import torch.nn as nn


c:\Users\shang\Programs\anaconda3\Lib\site-packages\torch\utils\_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


Defining functions

In [2]:
def f_convert_and_normalize_audio(
    dir_source="audio_input",
    dir_wav="audio_wav",
    dir_target="audio_output",
    target_sr=44100,
    subtype="PCM_16",
    mono=True
):
    """
    Converts audio files from multiple formats to WAV, then normalizes them
    to mono, 44.1kHz sample rate, and 16-bit PCM.
    Frees up space by deleting intermediate WAVs after normalization.
    """
    # Create directories if needed
   
    os.makedirs(dir_wav, exist_ok=True)
    os.makedirs(dir_target, exist_ok=True)

    # Step 1: Converting all formats to WAV.
    for file_name in os.listdir(dir_source):
        if file_name.lower().endswith((
            ".mp3", ".ogg", ".flac", ".aac", ".m4a", ".wma",
            ".aif", ".aiff", ".mp4", ".mpga", ".wav"
        )):
            input_path = os.path.join(dir_source, file_name)
            output_path = os.path.join(dir_wav, os.path.splitext(file_name)[0] + ".wav")

            try:
                audio = AudioSegment.from_file(input_path)
                audio.export(output_path, format="wav")
            except Exception as e:
                print(f"Failed to convert {file_name}: {e}")

    # Step 2: Normalizing WAVs to mono, 44.1kHz, 16-bit PCM
    for file_name in os.listdir(dir_wav):
        input_path = os.path.join(dir_wav, file_name)
        output_path = Path(dir_target) / file_name

        try:
            samples, sr = librosa.load(input_path, sr=target_sr, mono=mono)
            sf.write(output_path, samples, samplerate=target_sr, subtype=subtype)

            # Delete intermediate .wav to save space
            os.remove(input_path)

        except Exception as e:
            print(f"Could not normalize {file_name}: {e}")

    print("All audio files processed and saved to:", dir_target)

    # Clean up intermediary folders if empty
    try:
        os.rmdir(dir_wav)
        print(f"Removed temporary folder: {dir_wav}")
    except OSError:
        print(f"Could not remove {dir_wav} (not empty or in use).")


In [3]:
def f_audio_clean(y, threshold_db=-35):
    """
    Determines if the audio chunk contains meaningful sound based on RMS energy.
    """
    if not isinstance(y, np.ndarray):
        y = np.array(y).astype(np.float32)
    rms = librosa.feature.rms(y=y)[0]
    db = librosa.amplitude_to_db(rms, ref=np.max)
    percent_below = np.mean(db < threshold_db)
    return percent_below < 0.9


In [4]:
def f_split(input_folder="audio_output", duration=3):

    """
    Splits pre-normalized WAV files into chunks of fixed duration,
    padding the final chunk with silence if it's too short.
    Moves non-meaningful audio files to a _skipped folder.
    Deletes input files and folder after processing.

    """
    output_folder_split = f"{input_folder}_{duration}" # Create output folder name
    output_folder_skipped = f"{output_folder_split}_skipped" # Create skipped folder name
    os.makedirs(output_folder_split, exist_ok=True)
    os.makedirs(output_folder_skipped, exist_ok=True)

    chunk_length_ms = duration * 1000

    for audio_file in os.listdir(input_folder):
        if audio_file.lower().endswith(".wav"):
            input_path = os.path.join(input_folder, audio_file)
            audio = AudioSegment.from_wav(input_path)
            total_length_ms = len(audio)
            num_chunks = math.ceil(total_length_ms / chunk_length_ms)
            base_name = os.path.splitext(audio_file)[0]

            for i in range(num_chunks):
                start_ms = i * chunk_length_ms
                end_ms = start_ms + chunk_length_ms
                chunk_audio = audio[start_ms:end_ms]

                # Pad if needed
                if len(chunk_audio) < chunk_length_ms:
                    padding = AudioSegment.silent(duration=chunk_length_ms - len(chunk_audio))
                    chunk_audio += padding
                chunk_audio = chunk_audio[:chunk_length_ms]

                # Calculate time boundaries in seconds
                start_sec = int(start_ms / 1000)
                end_sec = int(start_sec + duration)

                chunk_filename = os.path.join(
                    output_folder_split,
                    f"{base_name}_{start_sec:03d}_{end_sec:03d}.wav"
                )

                # Export first, then check meaningfulness
                chunk_audio.export(chunk_filename, format="wav")
                y = np.array(chunk_audio.get_array_of_samples()).astype(np.float32)

                if not f_audio_clean(y):
                    os.rename(chunk_filename, os.path.join(output_folder_skipped, os.path.basename(chunk_filename)))
            # Delete the processed file
            os.remove(input_path)

    # Delete the input folder after all files are processed
    os.rmdir(input_folder)

    print(f"Split and cleaned audio into {output_folder_split}")
    print(f"Skipped audio saved in {output_folder_skipped}")
    print(f"Deleted folder: {input_folder}")

In [5]:
def process_audio_files(input_folder, duration=3): 
    """
    Full audio processing:
    1. Convert and normalize to WAV format.
    2. Split into chunks of specified duration and filter silence.
    """
    f_convert_and_normalize_audio(dir_source=input_folder)
    f_split(input_folder="audio_output", duration=duration)

In [6]:
process_audio_files("Low_2", duration=3)

Failed to convert PSL7_20240510_060000.WAV: Decoding failed. ffmpeg returned error code: 4294967274

Output from ffmpeg/avlib:

ffmpeg version 2025-03-31-git-35c091f4b7-full_build-www.gyan.dev Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 14.2.0 (Rev1, Built by MSYS2 project)
  configuration: --enable-gpl --enable-version3 --enable-static --disable-w32threads --disable-autodetect --enable-fontconfig --enable-iconv --enable-gnutls --enable-lcms2 --enable-libxml2 --enable-gmp --enable-bzlib --enable-lzma --enable-libsnappy --enable-zlib --enable-librist --enable-libsrt --enable-libssh --enable-libzmq --enable-avisynth --enable-libbluray --enable-libcaca --enable-libdvdnav --enable-libdvdread --enable-sdl2 --enable-libaribb24 --enable-libaribcaption --enable-libdav1d --enable-libdavs2 --enable-libopenjpeg --enable-libquirc --enable-libuavs3d --enable-libxevd --enable-libzvbi --enable-libqrencode --enable-librav1e --enable-libsvtav1 --enable-libvvenc --enable-libwebp --ena

In [ ]:
# def move_silent_chunks(folder_with_chunks, threshold_db=-35):
#     """
#     Moves silent chunks from a folder into a '_skipped' subfolder based on RMS threshold.
#     """
#     skipped_folder = f"{folder_with_chunks}_skipped"
#     os.makedirs(skipped_folder, exist_ok=True)

#     for file_name in os.listdir(folder_with_chunks):
#         if file_name.lower().endswith(".wav"):
#             file_path = os.path.join(folder_with_chunks, file_name)

#             try:
#                 audio = AudioSegment.from_wav(file_path)
#                 y = np.array(audio.get_array_of_samples()).astype(np.float32)

#                 if not f_audio_clean(y, threshold_db=threshold_db):
#                     os.rename(file_path, os.path.join(skipped_folder, file_name))
#                     print(f"Moved to skipped: {file_name}")

#             except Exception as e:
#                 print(f"Error processing {file_name}: {e}")

#     print(f"Skipped audio saved in {skipped_folder}")